# Brawl Stars — CS156 First Pipeline
Question: Can team compositions predict wins better than baselines? Binary now; three-class softmax requires genuine draws. Data: 25 personal Brawl Ball matches on Pinhole Punt.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.features import normalize_names,validate_matches,vocabulary,encode
from src.models import evaluate
df=normalize_names(pd.read_csv(ROOT/'data/raw_matches.csv'))
assert validate_matches(df)
print('Matches:',len(df),'Brawlers:',len(vocabulary(df)),'Wins:',df.Blue_Win.sum())
display(df.head())


## Data and feature construction
Manually transcribed screenshots → Google Sheet → immutable CSV. Features exclude results and trophy changes. Signed feature x_j is +1 on Blue only, -1 on Red only, 0 absent or on both. Separate-side binary features are compared. Probabilities in the binary case use sigmoid; multiclass softmax will require actual draws.


In [ ]:
freq=pd.Series(df[['Blue_1','Blue_2','Blue_3','Red_1','Red_2','Red_3']].to_numpy().ravel()).value_counts()
freq.head(15).plot.bar(title='Brawler appearances');plt.tight_layout();plt.show()
print('Signed feature matrix:',encode(df,vocabulary(df),'signed').shape)


## Four comparison conditions
Class-prior baseline; first Blue slot proxy; signed-composition logistic regression; side-separated logistic regression. Five stratified folds create out-of-fold probabilities. Main metric is log loss; supplementary scores are Brier, accuracy and balanced accuracy. Small sample sizes make results exploratory.


In [ ]:
results=evaluate(df)
metrics=pd.DataFrame({k:{m:r[m] for m in ('accuracy','balanced_accuracy','log_loss','brier')} for k,r in results.items()}).T
display(metrics.sort_values('log_loss'))
metrics[['log_loss','brier']].plot.bar(title='Out-of-fold probabilities');plt.xticks(rotation=25,ha='right');plt.tight_layout();plt.show()


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
pred=results['signed_composition']
ConfusionMatrixDisplay.from_predictions(df.Blue_Win,pred['predicted'],display_labels=['Loss','Win']);plt.show()
plt.scatter(range(len(df)),pred['probabilities'],c=df.Blue_Win,cmap='coolwarm');plt.axhline(.5,ls='--',color='gray');plt.ylabel('Out-of-fold P(Win)');plt.xlabel('Match');plt.ylim(0,1);plt.show()


## Findings, limitations, and next steps
Interpret the measured scores relative to baselines, not as proof of causal draft strength. Twenty-five matches, one map and player context, repeated brawlers and no draws imply high uncertainty. Future work: verified match ID, timestamp, patch, personal brawler, more diverse games, observed draws, three-class multinomial logistic regression and independent players/patch evaluation.
## References
CS156 Sessions 2, 6, and 9; scikit-learn LogisticRegression, StratifiedKFold and metrics documentation; Google Sheet source linked in README.
